In [1]:
!pip install -q faiss-cpu sentence-transformers pypdf transformers sentencepiece accelerate gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 33.0 MB/s eta 0:00:00


In [2]:
from google.colab import files

uploaded = files.upload()
uploaded_filenames = list(uploaded.keys())

print("Uploaded files:", uploaded_filenames)

Saving CORALGOLDRESOURCES,LTD_05_28_2020-EX-4.1-CONSULTING AGREEMENT.PDF to CORALGOLDRESOURCES,LTD_05_28_2020-EX-4.1-CONSULTING AGREEMENT.PDF
Uploaded files: ['CORALGOLDRESOURCES,LTD_05_28_2020-EX-4.1-CONSULTING AGREEMENT.PDF']


In [3]:
from pypdf import PdfReader

documents = []

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text_parts = []

    for page in reader.pages:
        text = page.extract_text()
        if text:
            text_parts.append(text.strip())

    return "\n".join(text_parts)

def extract_text_from_txt(txt_path):
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

for filename in uploaded_filenames:
    full_text = ""

    try:
        if filename.lower().endswith(".pdf"):
            full_text = extract_text_from_pdf(filename)
        elif filename.lower().endswith(".txt"):
            full_text = extract_text_from_txt(filename)
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        continue

    if full_text.strip():
        documents.append({
            "text": full_text,
            "metadata": {
                "source": filename
            }
        })

print("Documents loaded:", len(documents))
print(documents[0]["metadata"] if documents else "No documents found")

Documents loaded: 1
{'source': 'CORALGOLDRESOURCES,LTD_05_28_2020-EX-4.1-CONSULTING AGREEMENT.PDF'}


In [4]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    text = " ".join(text.split())

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

chunked_docs = []

for doc in documents:
    chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_docs.append({
            "text": chunk,
            "metadata": {
                **doc["metadata"],
                "chunk_id": i
            }
        })

print("Chunks created:", len(chunked_docs))
print(chunked_docs[0]["metadata"] if chunked_docs else "No chunks")

Chunks created: 38
{'source': 'CORALGOLDRESOURCES,LTD_05_28_2020-EX-4.1-CONSULTING AGREEMENT.PDF', 'chunk_id': 0}


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = [doc["text"] for doc in chunked_docs]
embeddings = embedding_model.encode(texts, normalize_embeddings=True)

print("Embeddings created:", len(embeddings))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created: 38


In [6]:
import faiss

embedding_matrix = np.array(embeddings).astype("float32")
dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

print("FAISS index built with", index.ntotal, "vectors")

FAISS index built with 38 vectors


In [7]:
def retrieve(query, top_k=5):
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        results.append({
            "text": chunked_docs[idx]["text"],
            "metadata": chunked_docs[idx]["metadata"],
            "distance": float(dist)
        })
    return results

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm = llm.to(device)

print("Loaded on:", device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded on: cuda


In [9]:
def generate_answer(query, context, max_new_tokens=180):
    q = query.lower().strip()

    if "summarize" in q or "summary" in q:
        prompt = f"""
You are a helpful document assistant.

Summarize the following document content clearly and completely.
Focus on the main topics, key points, and important conclusions.
Do not invent information.

Context:
{context}

Summary:
""".strip()

    elif "compare" in q:
        prompt = f"""
You are a helpful document assistant.

Compare the relevant information in the context and explain similarities or differences clearly.
Use only the provided context.
If comparison is not possible, say: I don't know.

Context:
{context}

Question:
{query}

Comparison:
""".strip()

    else:
        prompt = f"""
You are a helpful document question-answering assistant.

Answer the question using ONLY the context.
Give a clear and slightly detailed answer.
If the answer is yes/no, explain why using the context.
If the answer is not found, say exactly: I don't know.

Context:
{context}

Question:
{query}

Answer:
""".strip()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [10]:
def ask(query, top_k=8):
    results = retrieve(query, top_k=top_k)

    context = "\n\n".join([r["text"] for r in results])
    answer = generate_answer(query, context)

    sources = []
    for i, r in enumerate(results, 1):
        sources.append(
            {
                "label": f"[Chunk {i}] source={r['metadata'].get('source')} | chunk_id={r['metadata'].get('chunk_id')} | distance={r['distance']:.4f}",
                "text": r["text"][:500]
            }
        )

    return {
        "question": query,
        "answer": answer,
        "sources": sources,
        "results": results
    }

In [11]:
result = ask("Summarize the uploaded documents", top_k=5)

print("Question:", result["question"])
print("\nAnswer:\n", result["answer"])
print("\nSources:\n", result["sources"][:1200])

Question: Summarize the uploaded documents

Answer:
 6) shall be deemed to have been given or made and received on the day on which it was so faxed (or the next business day)

Sources:
 [{'label': '[Chunk 1] source=CORALGOLDRESOURCES,LTD_05_28_2020-EX-4.1-CONSULTING AGREEMENT.PDF | chunk_id=12 | distance=1.5867', 'text': 'Companies including maps, data, records, reports, technical studies, drill hole logs, calculations, opinions, charts, drawings, sketches, plans, documents, summaries, memoranda, analysis and all geological or technical information; (c) all information related to the properties, projects, facilities, equipment and other assets used in the business of the Company and the Affiliated Companies, and all information related to the exploration or development of (or potential exploration or development '}, {'label': '[Chunk 2] source=CORALGOLDRESOURCES,LTD_05_28_2020-EX-4.1-CONSULTING AGREEMENT.PDF | chunk_id=11 | distance=1.6725', 'text': ', discovery, strategy, method, idea

In [12]:
def hallucination_score(answer, results):
    context = " ".join([r["text"] for r in results]).lower()
    answer_words = set(answer.lower().split())

    overlap = len(answer_words & set(context.split()))
    score = overlap / max(len(answer_words), 1)

    return score


def confidence_label(score):
    if score < 0.3:
        return "Low confidence"
    elif score < 0.6:
        return "Medium confidence"
    return "High confidence"

In [13]:
def rag_ui(query, top_k):
    if not query.strip():
        return "Please enter a question.", "", ""

    result = ask(query, top_k=int(top_k))
    answer = result["answer"]

    score = hallucination_score(answer, result["results"])
    confidence = confidence_label(score)

    formatted_sources = "\n\n".join(
        [f"{s['label']}\n{s['text']}" for s in result["sources"]]
    )

    meta = f"Confidence: {confidence} | Grounding Score: {score:.2f}"

    return answer, formatted_sources, meta

In [14]:
import gradio as gr

sample_questions = [
    "Summarize the uploaded documents",
    "What are the main topics discussed?",
    "Give a concise summary of all uploaded files",
    "What important points are mentioned?",
    "Compare the uploaded documents",
]

with gr.Blocks(title="Universal Document RAG Assistant") as demo:
    gr.Markdown("# Universal Document RAG Assistant")
    gr.Markdown("Upload PDF or TXT files, then ask questions and get source-backed answers.")

    with gr.Row():
        query = gr.Textbox(
            label="Your Question",
            placeholder="Ask something about your uploaded documents..."
        )
        top_k = gr.Slider(
            minimum=1,
            maximum=10,
            value=8,
            step=1,
            label="Top-K Retrieval"
        )

    ask_btn = gr.Button("Ask")

    answer_box = gr.Textbox(label="Answer", lines=8)
    sources_box = gr.Textbox(label="Retrieved Sources", lines=18)
    meta_box = gr.Textbox(label="System Feedback", lines=2)

    ask_btn.click(
        fn=rag_ui,
        inputs=[query, top_k],
        outputs=[answer_box, sources_box, meta_box]
    )

    gr.Examples(
        examples=[[q, 5] for q in sample_questions],
        inputs=[query, top_k]
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6408899525d85da020.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6408899525d85da020.gradio.live


In [15]:
test_cases = [
    {"question": "What are the main topics discussed?"},
    {"question": "Summarize the uploaded documents"},
    {"question": "Give a concise summary of all uploaded files"},
]

def evaluate_project2():
    scores = []

    for test in test_cases:
        print("=" * 70)
        question = test["question"]

        result = ask(question, top_k=8)
        answer = result["answer"]
        score = hallucination_score(answer, result["results"])
        conf = confidence_label(score)

        print("Question:", question)
        print("Answer:", answer)
        print("Confidence:", conf)
        print("Grounding Score:", round(score, 2))

        scores.append(score)

    print("\nFINAL METRICS")
    print("Average Grounding Score:", round(sum(scores) / len(scores), 2))